In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

data = load_iris()
X, y = data.data, data.target
feature_names = data.feature_names
clf = KNeighborsClassifier(n_neighbors=3)

N_wolves, Max_Iter, N_features = 10, 20, X.shape[1]
Positions = np.random.rand(N_wolves, N_features)

Alpha_pos, Beta_pos, Delta_pos = np.zeros(N_features), np.zeros(N_features), np.zeros(N_features)
Alpha_score, Beta_score, Delta_score = float("inf"), float("inf"), float("inf")

for t in range(Max_Iter):
    a = 2 - t * (2 / Max_Iter)

    for i in range(N_wolves):
        Binary_Mask = Positions[i] > 0.5
        if not np.any(Binary_Mask):
            Binary_Mask[np.random.randint(N_features)] = True

        Accuracy = cross_val_score(clf, X[:, Binary_Mask], y, cv=3).mean()
        Fitness = 0.99 * (1 - Accuracy) + 0.01 * (np.sum(Binary_Mask) / N_features)

        if Fitness < Alpha_score:
            Delta_score, Delta_pos = Beta_score, Beta_pos.copy()
            Beta_score, Beta_pos = Alpha_score, Alpha_pos.copy()
            Alpha_score, Alpha_pos = Fitness, Positions[i].copy()
        elif Fitness < Beta_score:
            Delta_score, Delta_pos = Beta_score, Beta_pos.copy()
            Beta_score, Beta_pos = Fitness, Positions[i].copy()
        elif Fitness < Delta_score:
            Delta_score, Delta_pos = Fitness, Positions[i].copy()

    for i in range(N_wolves):
        for d in range(N_features):
            X1 = Alpha_pos[d] - (2 * a * np.random.rand() - a) * abs(2 * np.random.rand() * Alpha_pos[d] - Positions[i][d])
            X2 = Beta_pos[d] - (2 * a * np.random.rand() - a) * abs(2 * np.random.rand() * Beta_pos[d] - Positions[i][d])
            X3 = Delta_pos[d] - (2 * a * np.random.rand() - a) * abs(2 * np.random.rand() * Delta_pos[d] - Positions[i][d])
            Positions[i][d] = (X1 + X2 + X3) / 3

selected_mask = Alpha_pos > 0.5
selected_features = [feature_names[i] for i in range(N_features) if selected_mask[i]]
original_acc = cross_val_score(clf, X, y, cv=3).mean()
optimized_acc = cross_val_score(clf, X[:, selected_mask], y, cv=3).mean()

print("RESULTS")
print("-" * 40)
print("Original Feature Count  :", N_features)
print("Selected Feature Count  :", len(selected_features))
print("Features Kept           :", selected_features)
print("Original Model Accuracy :", round(original_acc * 100, 2), "%")
print("Optimized Model Accuracy:", round(optimized_acc * 100, 2), "%")

RESULTS
----------------------------------------
Original Feature Count  : 4
Selected Feature Count  : 3
Features Kept           : ['sepal length (cm)', 'petal length (cm)', 'petal width (cm)']
Original Model Accuracy : 97.33 %
Optimized Model Accuracy: 97.33 %
